> ## SOLUTION / ANSWER KEY &mdash; Lab 8.6
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-06-handoff.ipynb`](../lab-06-handoff.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 8.6 &mdash; Explicit Handoff (Capped)

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 25 min &nbsp;|&nbsp; **Day 4 &middot; Module 8 &mdash; Multi-Agent Collaboration &amp; Decision Making**

### What you'll do
- Let each agent return the next agent, or 'done'
- Walk the handoff path from a starting agent
- Cap total handoffs so two agents can't loop forever

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

> **Framework note:** these labs use the **real** LangChain (`langchain`, `langchain-core`, `langchain-ollama`). The **graded** cells assert only on the deterministic parts you build &mdash; routing, synthesis, tool wiring, agent structure &mdash; and never call an LLM, so the lab always verifies offline. Cells marked **Optional &mdash; run it for real** call a live local model (`ollama run llama3.2:1b`, or Groq) and self-skip if none is reachable. You are building the **multi-agent customer-service chatbot** &mdash; the client's Lab 4.2.

**Reference:** [Module 8 slides &mdash; Message passing & shared state](../../presentation/day4-module8-multi-agent-collaboration.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 8 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket
WORK = "/tmp/biaa-lab-08-06"
os.makedirs(WORK, exist_ok=True)
def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening -- the optional live cells self-skip when it isn't."""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False
print("Working dir:", WORK, "| Ollama reachable:", ollama_up())

## Concept
Agents coordinate by **explicit handoffs**: an agent signals *&ldquo;done&rdquo;* or *&ldquo;this needs
the tech agent&rdquo;*, and control passes on. The danger is a **handoff loop** &mdash; A hands to B,
B hands back to A, forever (deck slide 19) &mdash; so you **cap total handoffs**, exactly like
`max_iterations`. Explicit handoffs plus a cap are what keep a team from becoming a mob.

In [ ]:
# Each agent is a function returning the NEXT agent's name, or "done".
print("handoff: supervisor -> billing -> tech -> done")

## Your Turn
Complete `run_handoffs`: stop at `"done"`, otherwise follow the handoff, all under a cap.

In [ ]:
def run_handoffs(start, agents, max_handoffs=5):
    current, path = start, []
    for _ in range(max_handoffs):
        path.append(current)
        nxt = agents[current]()
        if nxt == "done":
            break
        current = nxt
    return path

AGENTS = {"supervisor": lambda: "billing", "billing": lambda: "tech", "tech": lambda: "done"}
LOOP   = {"a": lambda: "b", "b": lambda: "a"}   # two agents that would loop forever

try:
    print('path :', run_handoffs('supervisor', AGENTS))
    print('loop :', run_handoffs('a', LOOP, max_handoffs=4))
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("the handoff path is walked in order", lambda: run_handoffs("supervisor", AGENTS) == ["supervisor", "billing", "tech"])
expect_true("it stops at 'done'", lambda: run_handoffs("supervisor", AGENTS)[-1] == "tech")
expect_true("the cap stops an infinite loop", lambda: len(run_handoffs("a", LOOP, max_handoffs=4)) == 4)
expect_true("a lower cap stops sooner", lambda: len(run_handoffs("a", LOOP, max_handoffs=2)) == 2)
expect_true("an immediate 'done' yields a single step", lambda: run_handoffs("x", {"x": lambda: "done"}) == ["x"])

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

---
### Key takeaway
Explicit handoffs plus a cap are the multi-agent version of the agent loop with max_iterations. Without the cap, two polite agents hand back and forth forever.

[&#8592; All Module 8 labs](./index.html) &nbsp;&middot;&nbsp; [Module 8 slides](../../presentation/day4-module8-multi-agent-collaboration.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>